# Phase 1: Basic Cleaning


In [83]:
# 1.1 import raw data
import pandas as pd
import numpy as np

df = pd.read_csv(r'D:\chennai1-pg-price-predictor\Data\raw\chennai_pg_dataset.csv')

In [84]:
df.shape

(1661, 39)

In [85]:
# 1.2 Deduplication
before = df.shape[0]

# id + occupancy combination vachu dedup pannuthu, full row vachu illa
df = df.drop_duplicates(subset=['id', 'occupancy'], keep='first')

after = df.shape[0]
print(f"{before - after} duplicate rows removed")
print("Shape after dedup:", df.shape)


130 duplicate rows removed
Shape after dedup: (1531, 39)


In [86]:
# Drop Unnecessary Columns
drop_cols = [
    'id', 'title', 'address', 'total_bathrooms',
    'gate_closing_time', 'warden', 'cooking_allowed',
    'guardian_required', 'nonveg_allowed', 'smoking_allowed'
]

existing = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=existing)

print(f"Dropped columns: {existing}")
print("Shape after dropping columns:", df.shape)

Dropped columns: ['id', 'title', 'address', 'total_bathrooms', 'gate_closing_time', 'warden', 'cooking_allowed', 'guardian_required', 'nonveg_allowed', 'smoking_allowed']
Shape after dropping columns: (1531, 29)


In [87]:
# 1.4 Drop Redundant Food Columns
food_cols = ['breakfast', 'lunch', 'dinner']

existing = [c for c in food_cols if c in df.columns]
df = df.drop(columns=existing)

print(f"Dropped columns: {existing}")
print("Shape after dropping food columns:", df.shape)

Dropped columns: ['breakfast', 'lunch', 'dinner']
Shape after dropping food columns: (1531, 26)


In [88]:
# remove rpws
df = df.dropna(subset=['rent', 'deposit', 'occupancy', 'attached_bathroom'])
df = df[df['rent'] >= 1000]

df.shape


(1436, 26)

In [89]:
# Fix data types
amenity_cols = [
    'attached_bathroom', 'mess', 'wifi', 'laundry', 'power_backup',
    'refrigerator', 'common_tv', 'room_cleaning', 'room_ac',
    'room_cupboard', 'room_tv', 'room_geyser', 'room_bedding',
    'room_attached_bath'
]

# Check unique values before converting (safety check)
for col in amenity_cols:
    print(col, df[col].unique())


attached_bathroom [False True]
mess [nan False True]
wifi [nan True False]
laundry [nan True False]
power_backup [nan False True]
refrigerator [nan True False]
common_tv [nan True False]
room_cleaning [nan True False]
room_ac [False True nan]
room_cupboard [False True nan]
room_tv [False True nan]
room_geyser [False True nan]
room_bedding [False True nan]
room_attached_bath [False True nan]


In [90]:
# Missing values-a False nu fill pannunga (amenity available illa nu treat pannuthu)
df[amenity_cols] = df[amenity_cols].fillna(False)

# dtype-a bool aa convert pannunga
df[amenity_cols] = df[amenity_cols].astype(bool)

print("Shape after fixing amenity columns:", df.shape)
df[amenity_cols].dtypes

Shape after fixing amenity columns: (1436, 26)


C:\Users\phari\AppData\Local\Temp\ipykernel_24460\549647577.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[amenity_cols] = df[amenity_cols].fillna(False)


attached_bathroom     bool
mess                  bool
wifi                  bool
laundry               bool
power_backup          bool
refrigerator          bool
common_tv             bool
room_cleaning         bool
room_ac               bool
room_cupboard         bool
room_tv               bool
room_geyser           bool
room_bedding          bool
room_attached_bath    bool
dtype: object

In [91]:
# 1.7 — Replace -10 sentinel with NaN
df['transit_score'] = df['transit_score'].replace(-10, np.nan)

# Step Create a missingness indicator (before filling)
df['transit_score_missing'] = df['transit_score'].isna()

#  Fill missing values using locality-level median
df['transit_score'] = df.groupby('locality')['transit_score'].transform(
    lambda x: x.fillna(x.median())
)

overall_median = df['transit_score'].median()
df['transit_score'] = df['transit_score'].fillna(overall_median)

print("Remaining missing transit_score:", df['transit_score'].isna().sum())
print("transit_score_missing counts:\n", df['transit_score_missing'].value_counts())

Remaining missing transit_score: 0
transit_score_missing counts:
 transit_score_missing
False    759
True     677
Name: count, dtype: int64


d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [92]:
# 1.8 Create a missingness indicator (before filling)
df['lifestyle_score_missing'] = df['lifestyle_score'].isna()


df['lifestyle_score'] = df.groupby('locality')['lifestyle_score'].transform(
    lambda x: x.fillna(x.median())
)

overall_median_lifestyle = df['lifestyle_score'].median()
df['lifestyle_score'] = df['lifestyle_score'].fillna(overall_median_lifestyle)

print("Remaining missing lifestyle_score:", df['lifestyle_score'].isna().sum())
print("lifestyle_score_missing counts:\n", df['lifestyle_score_missing'].value_counts())

Remaining missing lifestyle_score: 0
lifestyle_score_missing counts:
 lifestyle_score_missing
False    762
True     674
Name: count, dtype: int64


d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\chennai1-pg-price-predictor\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [93]:
# Missing parking values-a 'none' category aa fill pannunga
df['parking'] = df['parking'].fillna('none')

print("parking value counts:\n", df['parking'].value_counts())

parking value counts:
 parking
Bike            1156
Bike and Car     137
none             112
Car               31
Name: count, dtype: int64


In [94]:
df['available_for'] = df['available_for'].replace('Both', 'Anyone')
print(df['available_for'].value_counts())

available_for
Anyone                  1307
Working Professional     121
Student                    8
Name: count, dtype: int64


In [95]:
# Final check
print("Final shape:", df.shape)
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicates:", df.duplicated().sum())

Final shape: (1436, 28)

Missing values:
 Series([], dtype: int64)

Duplicates: 0


# Phase 2: Preprocessing

## Split the data

In [96]:
from sklearn.model_selection import train_test_split

X = df.drop('rent', axis=1)
y = df['rent']

## Train Data

In [97]:
#First 80% Train + 20% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [98]:
# Then temporary 20%-a 50/50 split 
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)


# Identify Numerical & ategorical Features

###  why?
#####    -we  use different preprocessimg methods for categorical and numerical features.
   ##### -Numerical → Scaling / Transformation
 #####   -Categorical → Encoding

In [99]:
# cell 1 use X_train
numeric_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns

categorical_features = X_train.select_dtypes(
    include=['object']
).columns

# Ordinal encoding

In [100]:
from sklearn.preprocessing import OrdinalEncoder

occupancy_encoder = OrdinalEncoder(
    categories=[['SINGLE', 'DOUBLE', 'THREE', 'FOUR']]
)

X_train['occupancy_encoded'] = occupancy_encoder.fit_transform(
    X_train[['occupancy']]
)

X_val['occupancy_encoded'] = occupancy_encoder.transform(
    X_val[['occupancy']]
)

X_test['occupancy_encoded'] = occupancy_encoder.transform(
    X_test[['occupancy']]
)

In [101]:
print(X_train[['occupancy', 'occupancy_encoded']].head())
print(X_val[['occupancy', 'occupancy_encoded']].head())
print(X_test[['occupancy', 'occupancy_encoded']].head())

     occupancy  occupancy_encoded
982      THREE                2.0
962     SINGLE                0.0
340      THREE                2.0
1295     THREE                2.0
36      SINGLE                0.0
     occupancy  occupancy_encoded
608     SINGLE                0.0
301     DOUBLE                1.0
1438      FOUR                3.0
83      DOUBLE                1.0
242      THREE                2.0
     occupancy  occupancy_encoded
139       FOUR                3.0
265       FOUR                3.0
1650    DOUBLE                1.0
1451     THREE                2.0
1071    DOUBLE                1.0


In [102]:
X_train = X_train.drop(columns=['occupancy'])
X_val = X_val.drop(columns=['occupancy'])
X_test = X_test.drop(columns=['occupancy'])

In [103]:
# Identify categorical columns
categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print(categorical_cols)

['locality', 'gender', 'available_for', 'parking']


In [104]:
for col in ['gender', 'available_for', 'parking']:
    print(f"\n===== {col} =====")
    print(X_train[col].value_counts())
    print("Categories:", X_train[col].nunique())


===== gender =====
gender
MALE      576
FEMALE    540
BOTH       32
Name: count, dtype: int64
Categories: 3

===== available_for =====
available_for
Anyone                  1045
Working Professional      95
Student                    8
Name: count, dtype: int64
Categories: 3

===== parking =====
parking
Bike            933
Bike and Car    102
none             89
Car              24
Name: count, dtype: int64
Categories: 4


# One-Hot Encoding

In [105]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

In [106]:
cat_cols = ['gender','available_for','parking']

train_encoded = ohe.fit_transform(X_train[cat_cols])

In [107]:
val_encoded = ohe.transform(X_val[cat_cols])

test_encoded = ohe.transform(X_test[cat_cols])

In [108]:
encoded_cols = ohe.get_feature_names_out(cat_cols)

print(encoded_cols)

['gender_BOTH' 'gender_FEMALE' 'gender_MALE' 'available_for_Anyone'
 'available_for_Student' 'available_for_Working Professional'
 'parking_Bike' 'parking_Bike and Car' 'parking_Car' 'parking_none']


In [109]:
train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_cols,
    index=X_train.index
)

val_encoded_df = pd.DataFrame(
    val_encoded,
    columns=encoded_cols,
    index=X_val.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_cols,
    index=X_test.index
)

In [110]:
# remove the original columns
X_train = X_train.drop(columns=cat_cols)
X_val = X_val.drop(columns=cat_cols)
X_test = X_test.drop(columns=cat_cols)

In [111]:
# add the encoded columns
X_train = pd.concat([X_train, train_encoded_df], axis=1)
X_val = pd.concat([X_val, val_encoded_df], axis=1)
X_test = pd.concat([X_test, test_encoded_df], axis=1)

In [112]:
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (1148, 34)
Validation shape: (144, 34)
Test shape: (144, 34)


# Smoothed Target Encoding

In [113]:
locality_stats = X_train.copy()
locality_stats['rent'] = y_train

locality_stats = locality_stats.groupby('locality')['rent'].agg(
    ['mean', 'count']
)

print(locality_stats.head(10))

                                       mean  count
locality                                          
Adambakkam                      6480.769231     13
Alandur                         7557.142857     14
Arcot Road-Kodambakkam          5450.000000      7
Ashok Nagar                     6550.000000      4
Balaji Nagar                    7722.222222      9
East Coast Road-Thiruvanmiyur   8694.444444     36
East Tambaram                   7460.000000     15
GST Road-Tambaram               8318.181818     33
Guindy                          7974.358974     39
Indian Institute Of Technology  6666.666667      3


In [114]:
global_mean = y_train.mean()

print("Global mean rent:", global_mean)

Global mean rent: 7638.965156794425


In [115]:
# smooth encoding
alpha = 10

locality_stats['smoothed_mean'] = (
    (locality_stats['count'] * locality_stats['mean']
     + alpha * global_mean)
    / (locality_stats['count'] + alpha)
)

print(locality_stats.head(10))

                                       mean  count  smoothed_mean
locality                                                         
Adambakkam                      6480.769231     13    6984.332677
Alandur                         7557.142857     14    7591.235482
Arcot Road-Kodambakkam          5450.000000      7    6737.626563
Ashok Nagar                     6550.000000      4    7327.832255
Balaji Nagar                    7722.222222      9    7678.402714
East Coast Road-Thiruvanmiyur   8694.444444     36    8464.992425
East Tambaram                   7460.000000     15    7531.586063
GST Road-Tambaram               8318.181818     33    8160.224455
Guindy                          7974.358974     39    7905.911256
Indian Institute Of Technology  6666.666667      3    7414.588582


In [116]:
locality_mapping = locality_stats['smoothed_mean'].to_dict()

X_train['locality_encoded'] = X_train['locality'].map(
    locality_mapping
)

In [117]:
X_val['locality_encoded'] = X_val['locality'].map(
    locality_mapping
)

X_val['locality_encoded'] = X_val['locality_encoded'].fillna(
    global_mean
)


In [118]:
X_test['locality_encoded'] = X_test['locality'].map(
    locality_mapping
)

X_test['locality_encoded'] = X_test['locality_encoded'].fillna(
    global_mean
)

In [119]:
# remove original locality
X_train = X_train.drop(columns=['locality'])
X_val = X_val.drop(columns=['locality'])
X_test = X_test.drop(columns=['locality'])

# Numerical Preprocessing

In [120]:
numerical_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

print(numerical_cols)

['latitude', 'longitude', 'transit_score', 'lifestyle_score', 'deposit', 'occupancy_encoded', 'gender_BOTH', 'gender_FEMALE', 'gender_MALE', 'available_for_Anyone', 'available_for_Student', 'available_for_Working Professional', 'parking_Bike', 'parking_Bike and Car', 'parking_Car', 'parking_none', 'locality_encoded']


## Scaling

In [121]:
numeric_cols = [
    'latitude',
    'longitude',
    'transit_score',
    'lifestyle_score',
    'deposit',
    'locality_encoded'
]

In [122]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [123]:
X_train[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

In [124]:
# validation
X_val[numeric_cols] = scaler.transform(
    X_val[numeric_cols]
)

# test
X_test[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)

In [125]:
print(X_train[numeric_cols].describe().T)

                   count          mean       std       min       25%  \
latitude          1148.0 -1.128327e-14  1.000436 -1.643354 -0.964540   
longitude         1148.0 -2.679080e-13  1.000436 -2.905781  0.060446   
transit_score     1148.0  4.951517e-17  1.000436 -4.404772 -0.700001   
lifestyle_score   1148.0 -1.237879e-16  1.000436 -5.122056 -0.504688   
deposit           1148.0  2.166289e-17  1.000436 -0.885619 -0.398702   
locality_encoded  1148.0 -2.258356e-15  1.000436 -3.284022 -0.398279   

                       50%       75%       max  
latitude          0.348304  0.512882  3.254983  
longitude         0.419055  0.536155  1.117908  
transit_score     0.411430  0.707811  1.448765  
lifestyle_score  -0.406446  0.674215  2.835536  
deposit          -0.398702 -0.073982  8.855804  
locality_encoded -0.147031  0.568176  2.176917  


In [126]:
print("Object columns:",
      X_train.select_dtypes(include=['object']).columns.tolist())

Object columns: []


# MODELING PHASE:

## Baseline Linear Regression

In [127]:
from sklearn.linear_model import LinearRegression

baseline_model = LinearRegression()

In [128]:
# Train the model use only X_train and y_train
baseline_model.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [129]:
y_val_pred = baseline_model.predict(X_val)
print("Actual:", y_val.values[:10])
print("Predicted:", y_val_pred[:10])

Actual: [ 8000.  6500.  6500.  6000. 13000. 16500.  8000. 15500.  6500.  8000.]
Predicted: [ 9389.6200383   8400.95232177  3901.24256112  6911.04073166
  7772.25021966 11208.84592723  7675.84474836 10429.83970747
  6390.19099877  6801.81246053]


# Evaluate the Baseline

We'll use the three metrics planned for this project:

MAE

RMSE

R²

In [130]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_val, y_val_pred)

rmse = np.sqrt(
    mean_squared_error(y_val, y_val_pred)
)

r2 = r2_score(y_val, y_val_pred)
print("Baseline Linear Regression")
print("--------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

Baseline Linear Regression
--------------------------
MAE : 1640.5332275498306
RMSE: 2606.5031846883494
R²  : 0.25008535888876504


# Random Forest Regression

In [131]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
#train
rf_model.fit(X_train, y_train)
# validation
y_val_pred_rf = rf_model.predict(X_val)

In [132]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_mae = mean_absolute_error(y_val, y_val_pred_rf)

rf_rmse = np.sqrt(
    mean_squared_error(y_val, y_val_pred_rf)
)

rf_r2 = r2_score(y_val, y_val_pred_rf)

print("Random Forest Regression")
print("------------------------")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

Random Forest Regression
------------------------
MAE : 1399.0975
RMSE: 2293.7498564686603
R²  : 0.4192524025920815


# Gradiant Boosting

In [133]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
# train
gb_model.fit(X_train, y_train)
# validation
y_val_pred_gb = gb_model.predict(X_val)

In [134]:
gb_mae = mean_absolute_error(y_val, y_val_pred_gb)

gb_rmse = np.sqrt(
    mean_squared_error(y_val, y_val_pred_gb)
)

gb_r2 = r2_score(y_val, y_val_pred_gb)

print("Gradient Boosting Regression")
print("----------------------------")
print("MAE :", gb_mae)
print("RMSE:", gb_rmse)
print("R²  :", gb_r2)

Gradient Boosting Regression
----------------------------
MAE : 1357.6813412180697
RMSE: 2117.895080303405
R²  : 0.5048871523226086


# Hyperoarameter Tuning

In [141]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4]
}

gb_tuning = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

gb_tuning.fit(X_train, y_train)

,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.03, 0.05, ...], 'max_depth': [2, 3, ...], 'n_estimators': [100, 200, ...]}"
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'squared_error'


In [ ]:
print("Best Parameters:")
print(gb_tuning.best_params_)

print("\nBest CV RMSE:")
print(-gb_tuning.best_score_)

Best Parameters:
{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100}

Best CV RMSE:
2532.2573284245314


# Comparision

In [152]:
# Tuned GB — calculate validation metrics (for comparison, not used as final model)
y_val_pred_tuned = gb_tuning.best_estimator_.predict(X_val)

tuned_mae = mean_absolute_error(y_val, y_val_pred_tuned)
tuned_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_tuned))
tuned_r2 = r2_score(y_val, y_val_pred_tuned)

# Build comparison table dynamically from actual variables
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'Gradient Boosting',
        'Tuned Gradient Boosting'
    ],
    'MAE': [mae, rf_mae, gb_mae, tuned_mae],
    'RMSE': [rmse, rf_rmse, gb_rmse, tuned_rmse],
    'R2': [r2, rf_r2, gb_r2, tuned_r2]
})

print(results)

                     Model          MAE         RMSE        R2
0        Linear Regression  1640.533228  2606.503185  0.250085
1            Random Forest  1399.097500  2293.749856  0.419252
2        Gradient Boosting  1357.681341  2117.895080  0.504887
3  Tuned Gradient Boosting  1374.561458  2252.139029  0.440132


# Final Test Evaluation

In [151]:
# Predict On Test
y_test_pred_gb = gb_model.predict(X_test)

# Calculate Final Metrics
test_mae = mean_absolute_error(y_test, y_test_pred_gb)

test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred_gb)
)

test_r2 = r2_score(y_test, y_test_pred_gb)
# Print The Final Result
print("FINAL GRADIENT BOOSTING - TEST RESULTS")
print("--------------------------------------")
print("MAE :", test_mae)
print("RMSE:", test_rmse)
print("R²  :", test_r2)

FINAL GRADIENT BOOSTING - TEST RESULTS
--------------------------------------
MAE : 1245.6556895511048
RMSE: 2200.8479011637514
R²  : 0.4624529248619821
